In [1]:
import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

train.head()

import glob

matches = glob.glob("/kaggle/**/*.whl", recursive=True)
matches = [m for m in matches if "bitsandbytes" in m.lower()]
print(matches)

[]


In [2]:
import site

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)
import os
import kagglehub
import mamba_ssm
import torch
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Make sure the NVIDIA utility path is visible if needed
site.addsitedir("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script")
site.addsitedir("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/")

# Configuration
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32  # Can be set to a maximum of 32
OFFLOAD_DIR = "/kaggle/working/offload"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(OFFLOAD_DIR, exist_ok=True)

# -----------------------------
# Tokenizer
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Many causal LMs do not define a pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# -----------------------------
# Model
# -----------------------------
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("Model loaded successfully.")
# Disable cache during training
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# -----------------------------
# LoRA
# -----------------------------
# Initialize LoRA Adapter
print(f"Initializing LoRA adapter with rank={LORA_RANK}...")
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.train()
model.print_trainable_parameters()


# YOUR CODE HERE
# --------------
# model.train() 
# --------------
# (Within YOUR CODE HERE)
# -----------------------------
# Dataset
# -----------------------------
records = train.to_dicts()

class PromptAnswerDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]

def collate_fn(batch):
    input_tensors = []
    label_tensors = []

    for row in batch:
        prompt = row["prompt"]
        answer = row["answer"]

        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]

        # Add EOS so targets have a natural stop point
        ids = prompt_ids + answer_ids + [tokenizer.eos_token_id]
        labels = [-100] * len(prompt_ids) + answer_ids + [tokenizer.eos_token_id]

        input_tensors.append(torch.tensor(ids, dtype=torch.long))
        label_tensors.append(torch.tensor(labels, dtype=torch.long))

    input_ids = pad_sequence(
        input_tensors,
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )
    labels = pad_sequence(
        label_tensors,
        batch_first=True,
        padding_value=-100
    )
    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

dataset = PromptAnswerDataset(records)
loader = DataLoader(
    dataset,
    batch_size=1,   # safer for this huge model
    shuffle=True,
    collate_fn=collate_fn,
)

# -----------------------------
# Optimizer
# -----------------------------
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-5
)

def get_input_device(m):
    # With device_map="auto", model.device is not reliable for training input placement
    for p in m.parameters():
        if p.device.type != "meta":
            return p.device
    return torch.device("cpu")

input_device = get_input_device(model)
print("Input device:", input_device)

# -----------------------------
# Training loop
# -----------------------------
for step, batch in enumerate(loader):
    batch = {k: v.to(input_device) for k, v in batch.items()}

    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    if step % 10 == 0:
        print(f"step={step}, loss={loss.item():.4f}")

# Save Adapter
print(f"Saving adapter to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)
`torch_dtype` is deprecated! Use `dtype` instead!


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')